# Wildlife YOLO11l → ONNX → Hailo-8 HEF

Bu notebook Raspberry Pi 5 + **AI HAT+ 26 TOPS (Hailo-8)** için hazırlanmıştır.

Hedef sınıflar:

| ID | Sınıf | Türkçe |
|---:|---|---|
| 0 | `bear` | ayı |
| 1 | `boar` | domuz / yaban domuzu |
| 2 | `deer` | geyik |
| 3 | `wolf` | kurt |
| 4 | `cow` | inek |
| 5 | `person` | insan |
| 6 | `dog` | köpek (kurtla karışmayı azaltmak için) |

Bu notebook şunları yapar:

1. Dataset ZIP dosyasını Google Drive'dan kullanır; her oturumda yeniden yükleme istemez.
2. Dataset ve etiketleri denetler.
3. Yeni eğitim, yarıda kesilen eğitimi birebir sürdürme ve tamamlanmış modeli ek epoch eğitme modları sunar.
4. Ayrı test kümesinde metrik, confusion matrix ve tahmin görselleri üretir.
5. Tek resim/klasör üzerinde görsel test yapar.
6. `best.pt` ve `best.onnx` üretir.
7. Uyumlu Hailo DFC kurulmuşsa doğrudan Hailo-8 `HEF` üretir; değilse Docker derlemesi için paket hazırlar.

> GPU aç: **Runtime → Change runtime type → T4 GPU** (veya daha güçlü GPU).

## Başlamadan önce: GPU seçimi

Colab içinde GPU şu menüden seçilir:

1. Notebook'u aç.
2. Üst menüden **Runtime** seç.
3. **Change runtime type** seç.
4. **Hardware accelerator** bölümünü `GPU` yap.
5. Varsa **GPU class / Runtime shape** kısmından seçim yap:

   ```text
   T4 GPU
   L4 GPU
   A100 GPU
   ```

L4 ve A100 genellikle ücretli compute unit ve anlık donanım uygunluğu gerektirir. Bağlandıktan sonra gerçekten hangi GPU'nun verildiğini ilk kod hücresindeki `nvidia-smi` çıktısından kontrol et.

## Bounding-box etiketli detection dataset kaynakları

Aşağıdaki kaynaklarda nesne kutusu içeren detection datasetleri bulunur:

- [Open Images V7](https://docs.ultralytics.com/datasets/detect/open-images-v7): `person`, `dog`, `cattle`, `bear`, `deer` ve benzeri genel sınıflar için güçlü başlangıç kaynağı.
- [LILA BC Camera Traps](https://lila.science/datasets/): gerçek fotokapan, gece/IR ve boş orman görüntüleri. Her alt datasette bounding box bulunmadığından açıklamasında **bounding boxes** yazanları seç.
- [ENA24 Detection](https://lila.science/datasets/ena24detection/): fotokapan görüntüleri ve bounding box içeren LILA dataseti.
- [Roboflow Universe](https://universe.roboflow.com/): `class:bear`, `class:wild boar`, `class:deer`, `class:wolf`, `class:cattle`, `class:person`, `class:dog` şeklinde ara ve `YOLOv8/YOLO11` biçiminde dışa aktar.
- [Kaggle Datasets](https://www.kaggle.com/datasets): yalnız `object detection`, `YOLO` veya `bounding box` olarak açıkça belirtilen datasetleri kullan.

Topluluk datasetlerindeki her kutuyu ve sınıfı kontrol et. `pig` ile `wild boar`, köpek ile kurt, Amerikan kara ayısı ile Türkiye'deki boz ayı aynı hedef değildir. Gerçek Camera Module 3 NoIR + 850 nm IR görüntüleri kamuya açık verilerin yerini tamamen tutamaz.

## Roboflow: çok sınıflı bir datasette yalnız tek sınıfı dışa aktarma

`Versions` sayfasında doğrudan bir sınıf seçme kutusu bulunmaz. Sınıf seçimi, **yeni sürüm oluşturma ekranının Preprocessing bölümünde** yapılır:

1. Universe'daki dataset başkasına aitse önce **Download Dataset → Fork/Import to Workspace** ile kendi çalışma alanına kopyala.
2. Kendi projende sol menüden **Versions** seç.
3. **Generate New Version** seç.
4. Yeni sürüm ekranında **Preprocessing → Add Preprocessing Step → Modify Classes** seç.
5. İstenen sınıfı koru; diğer bütün sınıfları **Omit** yap.
6. Ardından **Filter Null** ekle. Böylece diğer sınıflar kaldırılınca etiketsiz kalan görüntüler dışarıda bırakılır.
7. **Generate** ile sürümü oluştur ve **Download Dataset → YOLOv11 → Download ZIP** yoluyla indir.

Örnek: Kaynakta `people`, `car` ve `house` varsa:

| Kaynak sınıf | Modify Classes işlemi |
|---|---|
| `people` | `person` olarak Remap/Override |
| `car` | Omit |
| `house` | Omit |

Bunun sonucu, `people` bulunan görüntülerde yalnız insan kutularının kaldığı ve sınıf adının `person` olduğu yeni bir sürümdür. Görüntüde hem insan hem araba varsa görüntü kalır fakat araba kutusu çıkarılır. Sadece araba/ev bulunan görüntüler `Filter Null` sayesinde çıkarılır.

### `Modify Classes` yeni sürüm ekranında görünmüyorsa

Önce projenin gerçekten kendi Workspace'ine fork edildiğini kontrol et; Universe'daki kaynak projeyi yerinde değiştiremezsin. Yine görünmüyorsa yalnızca **fork edilmiş çalışma kopyasında** şu yolu kullan:

1. **Classes & Tags → Modify Classes** aç.
2. İstenen sınıfın **Override** alanına yeni adını yaz.
3. İstenmeyen sınıfları sil/çıkar.
4. Sonra **Versions → Generate New Version** ekranında **Filter Null** uygula.

> Bu ikinci yol proje sınıflarını değiştirir ve geri alınması zor olabilir. Kaynak dataset üzerinde değil, yalnız fork edilmiş çalışma kopyasında kullan. Sadece mevcut sürümü etkileyen `Preprocessing → Modify Classes` yolu daha güvenlidir.

### Garip sınıf adlarını standartlaştırma

Aynı hedefi anlatan sınıfları tek İngilizce ada eşleştir:

```text
tanımlanan kişi → person
people          → person
human           → person
pedestrian      → person   # gerçekten yaya/insan kutusuysa
```

Birden fazla kaynak sınıfı aynı `person` adına Remap/Override etmek onları birleştirir. Ancak `rider`, `mannequin`, `statue` gibi anlamı farklı sınıfları görmeden `person` ile birleştirme. Uyum için sınıf adlarında İngilizce ASCII harfler, rakam ve gerekirse `-` kullan; boşluk ve Türkçe karakter kullanma.

### Roboflow sınıf adı ile YOLO class ID aynı şey değildir

Roboflow'da adı değiştirmek `.txt` içindeki sayıyı doğrudan seçmek anlamına gelmez. YOLO dışa aktarımında gerçek ID sırası, indirilen `data.yaml` dosyasındaki `names` listesinden belirlenir. Tek sınıflı bir export çoğunlukla şu şekilde gelir:

```yaml
names:
  0: person
```

Bu ZIP'i aşağıdaki yedi sınıflı ana datasete birleştirirken ID'yi ana sıraya dönüştürmek gerekir. Bu notebook'un sabit sırası:

```text
0 bear
1 boar
2 deer
3 wolf
4 cow
5 person
6 dog
```

Dolayısıyla tek sınıflı Roboflow exportundaki `0 person`, ana datasette **`5 person`** olmalıdır. `person` için `1` kullanma; bu projede `1`, `boar` sınıfıdır. Etiket düzenlerken satırın ilk karakterini değil, boşlukla ayrılmış ilk alan olan `class_id` değerini değiştir. Koordinatlara dokunma.

## Önerilen veri miktarı

Bunlar birbirinden gerçekten farklı görüntü hedefleridir. Aynı videodan çıkan yüzlerce benzer kare ayrı veri sayılmaz. Sayılarda yaklaşık `%20` sapma kabul edilebilir.

| Sınıf | Train/gündüz | Val/gündüz | Test/gündüz | Train/gece | Val/gece | Test/gece | Toplam |
|---|---:|---:|---:|---:|---:|---:|---:|
| Ayı `bear` | 576 | 72 | 72 | 384 | 48 | 48 | **1.200** |
| Yaban domuzu `boar` | 768 | 96 | 96 | 512 | 64 | 64 | **1.600** |
| Geyik `deer` | 768 | 96 | 96 | 512 | 64 | 64 | **1.600** |
| Kurt `wolf` | 864 | 108 | 108 | 576 | 72 | 72 | **1.800** |
| İnek `cow` | 576 | 72 | 72 | 384 | 48 | 48 | **1.200** |
| İnsan `person` | 720 | 90 | 90 | 480 | 60 | 60 | **1.500** |
| Köpek `dog` | 864 | 108 | 108 | 576 | 72 | 72 | **1.800** |
| Negatif/boş | 880 | 110 | 110 | 880 | 110 | 110 | **2.200** |
| **Toplam** | **6.016** | **752** | **752** | **4.304** | **538** | **538** | **12.900** |

`gece` sütunları, sonradan siyah-beyaza çevrilmiş gündüz fotoğraflarını değil, mümkün olduğunca gerçek IR/gece görüntülerini ifade eder.

## Hayvan bulunmayan fotoğraflar

Bunlar **negatif görüntülerdir** ve gereklidir.

Örnek:

```text
images/train/empty_forest_001.jpg
labels/train/empty_forest_001.txt
```

`empty_forest_001.txt` dosyası mevcut olacak ancak tamamen boş kalacak. Şunları kesinlikle yazma:

```text
background
-1
0 0 0 0 0
```

Validation ve test için de aynı sistem geçerlidir:

```text
images/val/empty_night_001.jpg
labels/val/empty_night_001.txt       # boş dosya

images/test/empty_night_002.jpg
labels/test/empty_night_002.txt      # boş dosya
```

Notebook denetimi her fotoğraf için aynı isimli `.txt` aradığı için boş etiket dosyasını oluşturmalısın.

### Negatif fotoğraflarda neler bulunmalı?

Sadece tamamen boş orman kullanma. Modeli zorlayan görüntüleri koy:

- Gece boş orman
- IR ışığıyla parlayan yapraklar
- Ağaç gövdeleri ve kütükler
- Kaya ve çalılar
- Yağmur, sis ve kar
- Örümcek ağı
- Kameraya yakın böcekler
- Rüzgârda hareket eden dallar
- Göz gibi parlayan yansımalar
- Araç farları
- Hayvan gölgesi
- Hedef olmayan kedi, kuş, at ve koyun gibi hayvanlar

Ancak görüntüde `dog`, `person` veya diğer yedi sınıftan biri bulunuyorsa etiketi boş bırakma; o nesneyi kutula.

### Kaç negatif görüntü?

Pozitif görüntülerin yaklaşık `%15–25`i kadar kullan. Önerilen toplam **2.000–2.500** negatif görüntüdür:

```text
train: 1.600–2.000
val:     200–250
test:    200–250
```

Negatiflerin en az yarısı gerçek gece/IR görüntüsü olsun.

## 0. Dataset'i yalnızca bir kez hazırla

Google Drive'da aşağıdaki dosyayı oluştur:

```text
MyDrive/WildlifeYOLO/wildlife_dataset.zip
```

ZIP'in içinde şu yapı bulunmalı:

```text
images/
  train/
  val/
  test/
labels/
  train/
  val/
  test/
data.yaml
```

Her resmin aynı isimli `.txt` YOLO etiketi olmalı. Hayvan bulunmayan negatif resimler için boş `.txt` dosyası kullanabilirsin.

Etiket satırı:

```text
class_id x_center y_center width height
```

Koordinatlar `0–1` aralığında olmalı. `train/val/test` ayrımını aynı video veya aynı hayvan karşılaşmasının kareleri farklı bölümlere düşmeyecek şekilde, **olay bazında** yap.

In [ ]:
# GPU ve çalışma ortamı
import os, platform, subprocess, sys

print("Python:", sys.version)
print("Platform:", platform.platform())
subprocess.run(["nvidia-smi"], check=False)

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.6.122+-x86_64-with-glibc2.35


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [ ]:
# Google Drive'ı bağla. Veriler, checkpoint'ler ve sonuçlar burada kalıcıdır.
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Tek ayar hücresi

Normal kullanımda yalnızca bu hücreyi değiştir.

`TRAIN_MODE` seçenekleri:

- `new`: Baştan yeni YOLO11l eğitimi başlatır.
- `resume`: Kesinti olmuş eğitimi `last.pt` ve optimizer durumu ile birebir sürdürür.
- `extend`: En son `best.pt` ağırlığını alıp ilave eğitim aşaması başlatır. Tamamlanan eğitime yeni epoch eklemek için bunu kullan.

In [ ]:
from pathlib import Path

# Drive proje alanı
DRIVE_ROOT = Path('/content/drive/MyDrive/WildlifeYOLO')
DATASET_ZIP = DRIVE_ROOT / 'wildlife_dataset.zip'
RUNS_DIR = DRIVE_ROOT / 'runs'
TESTS_DIR = DRIVE_ROOT / 'tests'
EXPORTS_DIR = DRIVE_ROOT / 'exports'
TEST_INPUTS_DIR = DRIVE_ROOT / 'test_inputs'
DFC_DIR = DRIVE_ROOT / 'hailo_dfc'

# Model ayarları
BASE_MODEL = 'yolo11l.pt'
IMAGE_SIZE = 640
TRAIN_MODE = 'new'       # 'new', 'resume' veya 'extend'
NEW_EPOCHS = 150         # new modunda toplam hedef
EXTRA_EPOCHS = 40        # extend modunda ilave eğitim aşaması
BATCH = -1               # GPU belleğine göre otomatik
PATIENCE = 30
WORKERS = 2
SEED = 42

CLASS_NAMES = ['bear', 'boar', 'deer', 'wolf', 'cow', 'person', 'dog']

for folder in [DRIVE_ROOT, RUNS_DIR, TESTS_DIR, EXPORTS_DIR,
               TEST_INPUTS_DIR, DFC_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

assert TRAIN_MODE in {'new', 'resume', 'extend'}
assert DATASET_ZIP.exists(), f'Dataset bulunamadı: {DATASET_ZIP}'

print('Dataset:', DATASET_ZIP)
print('Eğitim modu:', TRAIN_MODE)
print('Sınıflar:', CLASS_NAMES)

Dataset: /content/drive/MyDrive/WildlifeYOLO/wildlife_dataset.zip
Eğitim modu: new
Sınıflar: ['bear', 'boar', 'deer', 'wolf', 'cow', 'person', 'dog']


In [ ]:
# Ultralytics'i kur ve kullanılan sürümü Drive'a kaydet.
!pip -q install -U ultralytics pyyaml

import ultralytics, torch
print('Ultralytics:', ultralytics.__version__)
print('PyTorch:', torch.__version__)
print('CUDA kullanılabilir:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'GPU açık değil. Runtime ayarından GPU seç.'

(DRIVE_ROOT / 'environment.txt').write_text(
    f'python={sys.version}\n'
    f'ultralytics={ultralytics.__version__}\n'
    f'torch={torch.__version__}\n',
    encoding='utf-8'
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.9/63.9 kB 8.1 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics: 8.4.121
PyTorch: 2.11.0+cu128
CUDA kullanılabilir: True


97

## 2. Dataset'i yerel Colab diskine aç

ZIP Google Drive'da kalır. Her yeni Colab oturumunda hızlı eğitim için `/content` alanına açılır; tekrar internetten yüklenmez.

In [ ]:
import shutil, zipfile

LOCAL_EXTRACT = Path('/content/wildlife_dataset')
if LOCAL_EXTRACT.exists():
    shutil.rmtree(LOCAL_EXTRACT)
LOCAL_EXTRACT.mkdir(parents=True)

with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
    zf.extractall(LOCAL_EXTRACT)

# ZIP tek bir üst klasörle paketlenmiş olsa da doğru dataset kökünü bul.
candidates = []
for p in [LOCAL_EXTRACT, *[x for x in LOCAL_EXTRACT.rglob('*') if x.is_dir()]]:
    if (p / 'images' / 'train').is_dir() and (p / 'labels' / 'train').is_dir():
        candidates.append(p)

if not candidates:
    raise FileNotFoundError(
        'ZIP içinde images/train ve labels/train yapısı bulunamadı.'
    )

DATASET_ROOT = min(candidates, key=lambda p: len(p.parts))

def split_name(base):
    if (DATASET_ROOT / base / 'val').is_dir():
        return 'val'
    if (DATASET_ROOT / base / 'validation').is_dir():
        return 'validation'
    return None

VAL_NAME = split_name('images')
assert VAL_NAME is not None, 'images/val veya images/validation bulunamadı.'
assert (DATASET_ROOT / 'labels' / VAL_NAME).is_dir(), 'Validation labels bulunamadı.'
assert (DATASET_ROOT / 'images' / 'test').is_dir(), 'images/test bulunamadı.'
assert (DATASET_ROOT / 'labels' / 'test').is_dir(), 'labels/test bulunamadı.'

print('Dataset kökü:', DATASET_ROOT)

Dataset kökü: /content/wildlife_dataset


In [ ]:
# data.yaml üret
import yaml

DATA_YAML = Path('/content/wildlife_data.yaml')
yaml_data = {
    'path': str(DATASET_ROOT),
    'train': 'images/train',
    'val': f'images/{VAL_NAME}',
    'test': 'images/test',
    'nc': len(CLASS_NAMES),
    'names': {i: name for i, name in enumerate(CLASS_NAMES)},
}
DATA_YAML.write_text(yaml.safe_dump(yaml_data, sort_keys=False), encoding='utf-8')
shutil.copy2(DATA_YAML, DRIVE_ROOT / 'wildlife_data.yaml')
print(DATA_YAML.read_text())

path: /content/wildlife_dataset
train: images/train
val: images/val
test: images/test
nc: 7
names:
  0: bear
  1: boar
  2: deer
  3: wolf
  4: cow
  5: person
  6: dog



## 3. Dataset denetimi

Bu hücre eksik etiketleri, hatalı sınıf ID'lerini ve `0–1` dışında kalan bounding box değerlerini kontrol eder. Hata varsa eğitime geçme.

In [ ]:
from collections import Counter

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
report = {}
total_errors = []

for split in ['train', VAL_NAME, 'test']:
    image_dir = DATASET_ROOT / 'images' / split
    label_dir = DATASET_ROOT / 'labels' / split
    images = sorted(p for p in image_dir.rglob('*') if p.suffix.lower() in IMG_EXTS)
    counts = Counter()
    missing = 0

    for image_path in images:
        relative = image_path.relative_to(image_dir).with_suffix('.txt')
        label_path = label_dir / relative
        if not label_path.exists():
            missing += 1
            total_errors.append(f'Eksik etiket: {label_path}')
            continue

        for line_no, line in enumerate(label_path.read_text(encoding='utf-8').splitlines(), 1):
            if not line.strip():
                continue
            parts = line.split()
            if len(parts) != 5:
                total_errors.append(f'Hatalı sütun: {label_path}:{line_no}')
                continue
            try:
                class_id = int(parts[0])
                coords = [float(x) for x in parts[1:]]
            except ValueError:
                total_errors.append(f'Sayısal hata: {label_path}:{line_no}')
                continue
            if not 0 <= class_id < len(CLASS_NAMES):
                total_errors.append(f'Hatalı class ID: {label_path}:{line_no} -> {class_id}')
            if not all(0.0 <= value <= 1.0 for value in coords):
                total_errors.append(f'Koordinat 0-1 dışında: {label_path}:{line_no}')
            counts[class_id] += 1

    report[split] = {'images': len(images), 'missing_labels': missing, 'boxes': counts}

for split, info in report.items():
    print(f'\n[{split}] images={info["images"]}, missing_labels={info["missing_labels"]}')
    for idx, name in enumerate(CLASS_NAMES):
        print(f'  {idx} {name:>6}: {info["boxes"][idx]} kutu')

if total_errors:
    print('\nİlk 30 hata:')
    print('\n'.join(total_errors[:30]))
    raise ValueError(f'Dataset denetiminde {len(total_errors)} hata bulundu.')

print('\nDataset yapısal denetimi başarılı.')


[train] images=14135, missing_labels=0
  0   bear: 1651 kutu
  1   boar: 2340 kutu
  2   deer: 3033 kutu
  3   wolf: 2827 kutu
  4    cow: 2850 kutu
  5 person: 13170 kutu
  6    dog: 2895 kutu

[val] images=1767, missing_labels=0
  0   bear: 208 kutu
  1   boar: 258 kutu
  2   deer: 384 kutu
  3   wolf: 372 kutu
  4    cow: 335 kutu
  5 person: 1618 kutu
  6    dog: 328 kutu

[test] images=1767, missing_labels=0
  0   bear: 204 kutu
  1   boar: 286 kutu
  2   deer: 373 kutu
  3   wolf: 368 kutu
  4    cow: 394 kutu
  5 person: 1581 kutu
  6    dog: 407 kutu

Dataset yapısal denetimi başarılı.


## 4. Etiketleri görsel olarak kontrol et

Rastgele örneklerde kutu ve sınıf adlarını gösterir. Yapısal denetimden geçmek, kutuların doğru hayvana çizildiğini garanti etmez; bu hücreyi mutlaka incele.

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw

random.seed(SEED)
train_images = [p for p in (DATASET_ROOT / 'images' / 'train').rglob('*')
                if p.suffix.lower() in IMG_EXTS]
sample_images = random.sample(train_images, min(12, len(train_images)))

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
for ax, image_path in zip(axes.flat, sample_images):
    image = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(image)
    w, h = image.size
    relative = image_path.relative_to(DATASET_ROOT / 'images' / 'train').with_suffix('.txt')
    label_path = DATASET_ROOT / 'labels' / 'train' / relative
    for line in label_path.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        class_id, xc, yc, bw, bh = map(float, line.split())
        class_id = int(class_id)
        x1, y1 = (xc - bw / 2) * w, (yc - bh / 2) * h
        x2, y2 = (xc + bw / 2) * w, (yc + bh / 2) * h
        draw.rectangle((x1, y1, x2, y2), outline='red', width=max(2, w // 400))
        draw.text((x1 + 3, y1 + 3), CLASS_NAMES[class_id], fill='yellow')
    ax.imshow(image)
    ax.set_title(image_path.name)
    ax.axis('off')

for ax in axes.flat[len(sample_images):]:
    ax.axis('off')
plt.tight_layout()

Output hidden; open in https://colab.research.google.com to view.

## 5. Model eğitimi / devam ettirme

### Yeni eğitim

Ayar hücresinde:

```python
TRAIN_MODE = 'new'
```

### Colab kesildiyse birebir devam

```python
TRAIN_MODE = 'resume'
```

Bu mod `last.pt` içindeki epoch, optimizer ve scheduler durumunu sürdürür.

### Eğitim tamamlandı ama sonuç yetmediyse

```python
TRAIN_MODE = 'extend'
EXTRA_EPOCHS = 40
```

Bu mod en son `best.pt` ağırlığından yeni bir ince ayar aşaması başlatır. Her testten sonra istediğin kadar tekrarlanabilir; ancak validation/test kötüleşiyorsa daha fazla epoch değil, veri veya etiket düzeltmesi gerekir.

In [ ]:
from datetime import datetime
from ultralytics import YOLO

def newest_checkpoint(filename):
    files = list(RUNS_DIR.rglob(filename))
    if not files:
        raise FileNotFoundError(f'{filename} bulunamadı: {RUNS_DIR}')
    return max(files, key=lambda p: p.stat().st_mtime)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

if TRAIN_MODE == 'new':
    model = YOLO(BASE_MODEL)
    run_name = f'wildlife_yolo11l_{timestamp}'
    train_results = model.train(
        data=str(DATA_YAML),
        epochs=NEW_EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH,
        patience=PATIENCE,
        close_mosaic=15,
        workers=WORKERS,
        seed=SEED,
        pretrained=True,
        project=str(RUNS_DIR),
        name=run_name,
        plots=True,
        save=True,
    )

elif TRAIN_MODE == 'resume':
    checkpoint = newest_checkpoint('last.pt')
    print('Birebir devam checkpoint:', checkpoint)
    model = YOLO(str(checkpoint))
    train_results = model.train(resume=True)

else:  # extend
    checkpoint = newest_checkpoint('best.pt')
    print('İlave eğitim başlangıcı:', checkpoint)
    model = YOLO(str(checkpoint))
    run_name = f'wildlife_extend_{timestamp}'
    train_results = model.train(
        data=str(DATA_YAML),
        epochs=EXTRA_EPOCHS,
        imgsz=IMAGE_SIZE,
        batch=BATCH,
        patience=min(PATIENCE, EXTRA_EPOCHS),
        close_mosaic=max(5, min(15, EXTRA_EPOCHS // 4)),
        workers=WORKERS,
        seed=SEED,
        project=str(RUNS_DIR),
        name=run_name,
        plots=True,
        save=True,
    )

BEST_PT = newest_checkpoint('best.pt')
LAST_PT = newest_checkpoint('last.pt')
print('En güncel best.pt:', BEST_PT)
print('En güncel last.pt:', LAST_PT)

Ultralytics 8.4.121 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/wildlife_data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=wildlife_yolo11l_202

## 6. Ayrı test kümesinde gerçek değerlendirme

Eğitim sonuçlarına karar verirken yalnız genel mAP'e bakma. Özellikle `wolf`, `dog`, gece görüntüleri ve sınıf başına recall değerlerini kontrol et.

Bu hücre `test` bölümünü kullanır ve Drive'a confusion matrix dahil test çıktıları kaydeder.

In [ ]:
BEST_PT = newest_checkpoint('best.pt')
test_model = YOLO(str(BEST_PT))
test_name = f'test_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

metrics = test_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=IMAGE_SIZE,
    batch=8,
    conf=0.001,
    iou=0.60,
    plots=True,
    project=str(TESTS_DIR),
    name=test_name,
)

print('\nTest sonuçları')
print('mAP50-95:', float(metrics.box.map))
print('mAP50:', float(metrics.box.map50))
print('mAP75:', float(metrics.box.map75))
print('\nSınıf bazında mAP50-95:')
for idx, value in enumerate(metrics.box.maps):
    print(f'{idx} {CLASS_NAMES[idx]:>6}: {float(value):.4f}')
print('Test çıktıları:', TESTS_DIR / test_name)

# Sonraki eğitimlerle karşılaştırmak için test özetini kalıcı geçmişe ekle.
import json
TEST_HISTORY = DRIVE_ROOT / 'test_history.jsonl'
history_record = {
    'timestamp': datetime.now().isoformat(timespec='seconds'),
    'checkpoint': str(BEST_PT),
    'test_name': test_name,
    'map50_95': float(metrics.box.map),
    'map50': float(metrics.box.map50),
    'map75': float(metrics.box.map75),
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
    'per_class_map50_95': {
        CLASS_NAMES[i]: float(value) for i, value in enumerate(metrics.box.maps)
    },
}
with TEST_HISTORY.open('a', encoding='utf-8') as f:
    f.write(json.dumps(history_record, ensure_ascii=False) + '\n')
print('Test geçmişine kaydedildi:', TEST_HISTORY)

NameError: name 'newest_checkpoint' is not defined

In [ ]:
# Confusion matrix ve test eğrilerini göster
from IPython.display import display

test_output = TESTS_DIR / test_name
plot_files = [
    test_output / 'confusion_matrix_normalized.png',
    test_output / 'PR_curve.png',
    test_output / 'F1_curve.png',
]
for plot_file in plot_files:
    if plot_file.exists():
        print(plot_file.name)
        display(Image.open(plot_file))

## 6.1 İsteğe bağlı: öğrenme eğrisi, önceki testlerle karşılaştırma ve doygunluk analizi

Bu hücre yalnızca sen istediğinde çalıştırılır. Şunları üretir:

- Bütün `new/extend` eğitimlerinin birleşik mAP ve loss eğrileri
- Önceki test ile son test arasındaki değişim
- Overfitting işareti
- Eğrinin platoya yaklaşmasını gösteren **doygunluk göstergesi**

> Doygunluk `%90`, “model gerçeğin yüzde 90'ını öğrendi” anlamına gelmez. Yalnız mevcut dataset ve ayarlarla öğrenme eğrisinin düzleştiğine, daha fazla epoch'un düşük getiri sağlayabileceğine dair sezgisel göstergedir. Yeni ve daha kaliteli veri, erişilebilecek sınırı değiştirebilir.

In [ ]:
# İSTEĞE BAĞLI HÜCRE
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

result_files = sorted(RUNS_DIR.rglob('results.csv'), key=lambda p: p.stat().st_mtime)
if not result_files:
    raise FileNotFoundError(f'results.csv bulunamadı: {RUNS_DIR}')

frames = []
global_start = 0
for result_file in result_files:
    frame = pd.read_csv(result_file)
    frame.columns = [column.strip() for column in frame.columns]
    frame['run'] = result_file.parent.name
    frame['global_epoch'] = np.arange(global_start + 1, global_start + len(frame) + 1)
    global_start += len(frame)
    frames.append(frame)

history = pd.concat(frames, ignore_index=True)

def find_column(candidates):
    for candidate in candidates:
        if candidate in history.columns:
            return candidate
    raise KeyError(f'Beklenen sütun bulunamadı: {candidates}')

map_col = find_column(['metrics/mAP50-95(B)', 'metrics/mAP50-95'])
map50_col = find_column(['metrics/mAP50(B)', 'metrics/mAP50'])
precision_col = find_column(['metrics/precision(B)', 'metrics/precision'])
recall_col = find_column(['metrics/recall(B)', 'metrics/recall'])
train_box_col = find_column(['train/box_loss'])
val_box_col = find_column(['val/box_loss'])

curve = history[map_col].astype(float).to_numpy()
epochs = history['global_epoch'].to_numpy()
smooth_window = min(7, max(1, len(curve) // 10))
smooth = pd.Series(curve).rolling(smooth_window, min_periods=1).mean().to_numpy()
recent_window = min(20, max(5, len(curve) // 4))

recent_x = np.arange(recent_window, dtype=float)
recent_y = smooth[-recent_window:]
recent_slope = float(np.polyfit(recent_x, recent_y, 1)[0]) if recent_window > 1 else 0.0
recent_gain = float(recent_y[-1] - recent_y[0])
best_value = float(np.max(smooth))
current_value = float(smooth[-1])
epochs_since_best = int(len(smooth) - 1 - int(np.argmax(smooth)))

# İlk bölümdeki öğrenme hızını referans al.
early_window = min(20, max(5, len(curve) // 4))
early_y = smooth[:early_window]
early_slope = float(np.polyfit(np.arange(early_window), early_y, 1)[0]) if early_window > 1 else 0.0
positive_reference = max(early_slope, 0.001)
slope_ratio = max(0.0, recent_slope) / positive_reference
plateau_from_slope = 1.0 - float(np.clip(slope_ratio, 0.0, 1.0))
plateau_from_age = 1.0 - math.exp(-epochs_since_best / 8.0)
plateau_from_gain = 1.0 - float(np.clip(max(recent_gain, 0.0) / 0.02, 0.0, 1.0))

train_loss = history[train_box_col].astype(float).to_numpy()
val_loss = history[val_box_col].astype(float).to_numpy()
loss_w = min(recent_window, len(train_loss))
train_loss_slope = float(np.polyfit(np.arange(loss_w), train_loss[-loss_w:], 1)[0])
val_loss_slope = float(np.polyfit(np.arange(loss_w), val_loss[-loss_w:], 1)[0])
map_drop = best_value - current_value
overfit = bool(train_loss_slope < 0 and val_loss_slope > 0 and map_drop > 0.01)

saturation = 100.0 * (
    0.50 * plateau_from_slope +
    0.25 * plateau_from_age +
    0.25 * plateau_from_gain
)
saturation = float(np.clip(saturation, 0.0, 100.0))

# Kalıcı test geçmişini oku.
test_records = []
TEST_HISTORY = DRIVE_ROOT / 'test_history.jsonl'
if TEST_HISTORY.exists():
    for line in TEST_HISTORY.read_text(encoding='utf-8').splitlines():
        if line.strip():
            test_records.append(json.loads(line))

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

axes[0].plot(epochs, history[map50_col], alpha=0.35, label='mAP50')
axes[0].plot(epochs, curve, alpha=0.35, label='mAP50-95 ham')
axes[0].plot(epochs, smooth, linewidth=2.5, label='mAP50-95 yumuşatılmış')
axes[0].axhline(best_value, color='green', linestyle='--', alpha=0.6,
                label=f'En iyi {best_value:.3f}')
axes[0].set_title(f'Öğrenme eğrisi — doygunluk ≈ %{saturation:.0f}')
axes[0].set_xlabel('Birleşik epoch')
axes[0].set_ylabel('Metrik')
axes[0].grid(alpha=0.25)
axes[0].legend(fontsize=8)

axes[1].plot(epochs, train_loss, label='Train box loss')
axes[1].plot(epochs, val_loss, label='Validation box loss')
axes[1].set_title('Train / validation loss')
axes[1].set_xlabel('Birleşik epoch')
axes[1].grid(alpha=0.25)
axes[1].legend()

if test_records:
    test_x = np.arange(1, len(test_records) + 1)
    for key, label in [
        ('map50_95', 'Test mAP50-95'),
        ('map50', 'Test mAP50'),
        ('precision', 'Test precision'),
        ('recall', 'Test recall'),
    ]:
        axes[2].plot(test_x, [r[key] for r in test_records], marker='o', label=label)
    axes[2].set_xticks(test_x)
    axes[2].set_xlabel('Test çalıştırma sırası')
    axes[2].set_title('Önceki testlerle karşılaştırma')
    axes[2].grid(alpha=0.25)
    axes[2].legend(fontsize=8)
else:
    axes[2].text(0.5, 0.5, 'Henüz test geçmişi yok', ha='center', va='center')
    axes[2].set_axis_off()

plt.suptitle(
    f'Son {recent_window} epoch kazancı: {recent_gain:+.4f} | '
    f'Eğim: {recent_slope:+.6f}/epoch | En iyiden fark: {map_drop:.4f}',
    fontsize=11
)
plt.tight_layout()

analysis_dir = TESTS_DIR / f'learning_analysis_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
analysis_dir.mkdir(parents=True, exist_ok=True)
graph_path = analysis_dir / 'learning_and_test_comparison.png'
fig.savefig(graph_path, dpi=160, bbox_inches='tight')
plt.show()

print(f'\nDoygunluk/platoya yaklaşma göstergesi: %{saturation:.1f}')
print(f'Son {recent_window} epoch mAP50-95 değişimi: {recent_gain:+.4f}')
print(f'Yumuşatılmış en iyi mAP50-95: {best_value:.4f}')
print(f'Yumuşatılmış güncel mAP50-95: {current_value:.4f}')
print(f'En iyi değerden beri geçen epoch: {epochs_since_best}')
print('Overfitting işareti:', 'VAR' if overfit else 'belirgin değil')

if len(test_records) >= 2:
    previous, latest = test_records[-2], test_records[-1]
    print('\nÖnceki teste göre son değişim:')
    for key in ['map50_95', 'map50', 'precision', 'recall']:
        delta = latest[key] - previous[key]
        print(f'  {key:>10}: {previous[key]:.4f} → {latest[key]:.4f} ({delta:+.4f})')

if overfit:
    decision = 'DAHA FAZLA EPOCH ÖNERİLMEZ: validation kaybı artıyor ve mAP en iyi noktadan düşmüş.'
elif saturation >= 85 and recent_gain < 0.005:
    decision = 'PLATOYA ÇOK YAKIN: daha fazla epoch muhtemelen az kazandırır; yeni/çeşitli veri ekle.'
elif recent_gain >= 0.01 and recent_slope > 0:
    decision = 'MODEL HÂLÂ ÖĞRENİYOR: kontrollü 20–40 epoch extend denenebilir.'
else:
    decision = 'SINIRDA/BELİRSİZ: 20 epoch kontrollü extend yapıp bağımsız test değişimini karşılaştır.'

print('\nKarar:', decision)
print('Grafik kaydedildi:', graph_path)

summary = {
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'saturation_indicator_percent': saturation,
    'recent_window_epochs': recent_window,
    'recent_map50_95_gain': recent_gain,
    'recent_map50_95_slope_per_epoch': recent_slope,
    'best_smoothed_map50_95': best_value,
    'current_smoothed_map50_95': current_value,
    'epochs_since_best': epochs_since_best,
    'overfit_signal': overfit,
    'decision': decision,
    'warning': 'Doygunluk göstergesi gerçek öğrenme kapasitesinin kesin yüzdesi değildir.',
}
(analysis_dir / 'analysis_summary.json').write_text(
    json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8'
)

## 7. Kendi resimlerinle görsel test

Test etmek istediğin yeni resimleri Drive'daki şu klasöre koy:

```text
MyDrive/WildlifeYOLO/test_inputs/
```

Bu resimler eğitim dataset'inde bulunmamalı. Hücre kutulu sonuçları `tests/predictions_*` klasörüne kaydeder ve ekranda gösterir.

In [ ]:
BEST_PT = newest_checkpoint('best.pt')
predict_model = YOLO(str(BEST_PT))
predict_name = f'predictions_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

input_images = [p for p in TEST_INPUTS_DIR.rglob('*') if p.suffix.lower() in IMG_EXTS]
if not input_images:
    raise FileNotFoundError(f'Test resmi bulunamadı: {TEST_INPUTS_DIR}')

results = predict_model.predict(
    source=str(TEST_INPUTS_DIR),
    imgsz=IMAGE_SIZE,
    conf=0.20,
    iou=0.60,
    save=True,
    project=str(TESTS_DIR),
    name=predict_name,
)

predict_output = TESTS_DIR / predict_name
saved_images = [p for p in predict_output.rglob('*') if p.suffix.lower() in IMG_EXTS]
print('Tahminler:', predict_output)

for image_path in saved_images[:20]:
    display(Image.open(image_path))

## 8. PT ve ONNX export

Hailo derlemesine geçmeden önce seçilen `best.pt`, ONNX'e aktarılır ve Drive'a kopyalanır. Hailo için dinamik boyut kullanma; input sabit `640×640`, batch `1` olmalı.

In [ ]:
BEST_PT = newest_checkpoint('best.pt')
export_stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
EXPORT_DIR = EXPORTS_DIR / f'wildlife_{export_stamp}'
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy2(BEST_PT, EXPORT_DIR / 'best.pt')
shutil.copy2(DATA_YAML, EXPORT_DIR / 'wildlife_data.yaml')

export_model = YOLO(str(BEST_PT))
onnx_path = Path(export_model.export(
    format='onnx',
    imgsz=IMAGE_SIZE,
    opset=11,
    simplify=True,
    dynamic=False,
    batch=1,
))
shutil.copy2(onnx_path, EXPORT_DIR / 'best.onnx')
print('PT:', EXPORT_DIR / 'best.pt')
print('ONNX:', EXPORT_DIR / 'best.onnx')

## 9A. Doğrudan HEF export — tercih edilen yol

Hailo HEF derlemesi için Hailo Dataflow Compiler gerekir. Standart Colab bunu içermez.

1. Hailo Developer Zone'dan **Hailo-8 / DFC v3.x**, Colab'ın Python ve Linux sürümüyle uyumlu `.whl` dosyasını indir.
2. Dosyayı bir kez şu klasöre koy:

   ```text
   MyDrive/WildlifeYOLO/hailo_dfc/
   ```

3. Aşağıdaki kurulum hücresini çalıştır.

Uyumlu wheel yoksa bu bölüm çalışmaz. Hailo AI Software Suite Docker arşivini Colab içinde çalıştırmak güvenilir değildir; bu durumda **9B** paketini normal Ubuntu x86-64 Hailo Docker ortamında derle.

In [ ]:
# Hailo DFC wheel kur. Hata alırsan 9B yöntemini kullan.
import glob

dfc_wheels = sorted(DFC_DIR.glob('hailo_dataflow_compiler*.whl'))
if not dfc_wheels:
    raise FileNotFoundError(
        f'Uyumlu Hailo DFC wheel bulunamadı: {DFC_DIR}'
    )

DFC_WHEEL = dfc_wheels[-1]
print('Kurulacak DFC:', DFC_WHEEL.name)
subprocess.run([sys.executable, '-m', 'pip', 'install', str(DFC_WHEEL)], check=True)
print('DFC kuruldu. Import hatası olursa runtime yeniden başlatıp 0, 1, 8 ve 9A hücrelerini çalıştır.')

In [ ]:
# Doğrudan YOLO11l → Hailo-8 INT8 HEF
# conf=0.10 HEF içindeki NMS'e yazılır; uygulamada daha yüksek sınıf eşikleri uygulanabilir.
BEST_PT = newest_checkpoint('best.pt')
hef_model = YOLO(str(BEST_PT))

hef_output = Path(hef_model.export(
    format='hailo',
    name='hailo8',       # 26 TOPS Hailo-8; hailo8l kullanma
    imgsz=IMAGE_SIZE,
    data=str(DATA_YAML),
    fraction=1.0,
    conf=0.10,
    iou=0.60,
))

# Export bir klasör veya dosya döndürebilir; tamamını Drive'a kopyala.
direct_dir = EXPORT_DIR / 'hailo8_direct'
if direct_dir.exists():
    shutil.rmtree(direct_dir)
if hef_output.is_dir():
    shutil.copytree(hef_output, direct_dir)
else:
    direct_dir.mkdir(parents=True)
    shutil.copy2(hef_output, direct_dir / hef_output.name)

print('Hailo export:', direct_dir)
print('İçerik:', [p.name for p in direct_dir.iterdir()])

## 9B. Colab'da DFC çalışmazsa Docker derleme paketi

Bu hücre, mevcut Ubuntu/Hailo AI Software Suite Docker rehberinde kullanılacak paketi oluşturur:

- `best.pt`
- `best.onnx`
- `wildlife_data.yaml`
- Hailo INT8 calibration için temsilî resimler
- Örnek `compile_hailo8.sh`

Calibration görüntülerinin etikete ihtiyacı yoktur; fakat eğitim görüntülerinin etiketleri zorunludur. Calibration kümesi mümkün olduğunca gerçek Camera Module 3 NoIR + 850 nm IR ve gündüz kullanım dağılımını temsil etmelidir.

In [ ]:
# En fazla 2048 temsilî resmi train/val içinden kopyala.
CALIB_COUNT = 2048
calibration_dir = EXPORT_DIR / 'calibration_images'
if calibration_dir.exists():
    shutil.rmtree(calibration_dir)
calibration_dir.mkdir(parents=True)

all_calib_candidates = []
for split in ['train', VAL_NAME]:
    all_calib_candidates.extend(
        p for p in (DATASET_ROOT / 'images' / split).rglob('*')
        if p.suffix.lower() in IMG_EXTS
    )

random.Random(SEED).shuffle(all_calib_candidates)
selected = all_calib_candidates[:min(CALIB_COUNT, len(all_calib_candidates))]
for idx, src in enumerate(selected):
    shutil.copy2(src, calibration_dir / f'{idx:05d}_{src.name}')

compile_script = '\n'.join([
    '#!/usr/bin/env bash',
    'set -euo pipefail',
    '',
    '# Hailo AI Software Suite Docker içinde örnek kullanım.',
    '# Yol ve YAML adı kurulu Hailo Model Zoo sürümüne göre değişebilir.',
    'MODEL_ZOO_YAML=/local/workspace/hailo_model_zoo/hailo_model_zoo/cfg/networks/yolov11l.yaml',
    '',
    'hailomz compile \\',
    '  --ckpt /local/shared_with_docker/wildlife_export/best.onnx \\',
    '  --calib-path /local/shared_with_docker/wildlife_export/calibration_images \\',
    '  --yaml "$MODEL_ZOO_YAML" \\',
    '  --classes 7 \\',
    '  --hw-arch hailo8',
    '',
])
(EXPORT_DIR / 'compile_hailo8.sh').write_text(compile_script, encoding='utf-8')

readme = '\n'.join([
    'Wildlife YOLO11l Hailo-8 export paketi',
    '',
    'Model: YOLO11l detect',
    'Sınıf sayısı: 7',
    f'Sınıflar: {", ".join(CLASS_NAMES)}',
    f'Input: {IMAGE_SIZE}x{IMAGE_SIZE}, batch 1',
    'Hedef: hailo8 (26 TOPS)',
    '',
    '1. Bu klasörü Hailo Docker host shared_with_docker altında wildlife_export adıyla yerleştir.',
    '2. Hailo Model Zoo sürümünde yolov11l.yaml bulunduğunu doğrula.',
    '3. compile_hailo8.sh içindeki yolları kurulumuna göre kontrol et.',
    '4. Scripti Docker içinde çalıştır.',
    '',
    'Not: Farklı bir YOLO YAML (ör. yolov11n.yaml) kullanma. ONNX model YOLO11l olduğu için',
    'parser/YAML mimarisi de YOLO11l ile eşleşmelidir.',
    '',
])
(EXPORT_DIR / 'README_HAILO.txt').write_text(readme, encoding='utf-8')

archive = shutil.make_archive(str(EXPORT_DIR) + '_docker_package', 'zip', EXPORT_DIR)
print('Calibration resmi:', len(selected))
print('Docker paketi:', archive)

## 10. HEF'i cihazda doğrula

Üretilen `.hef` dosyasını Raspberry Pi 5'e kopyaladıktan sonra:

```bash
hailortcli fw-control identify
hailortcli parse-hef wildlife_yolo11l.hef
hailortcli benchmark wildlife_yolo11l.hef
```

Son karar yalnız `.pt` testine göre verilmemeli. Aynı gece/gündüz test görüntülerinde `.hef` sonuçlarını da ölç; INT8 quantization sonrasında özellikle kurt–köpek ayrımını kontrol et.

### Doğru tekrar döngüsü

```text
new eğitim
   ↓
test split + yeni saha resimleri
   ↓
Hata etiket kaynaklıysa dataset'i düzelt
   ↓
Eğitim yarıda kesildiyse resume
Eğitim tamamlandı fakat veri arttıysa extend
   ↓
PT testi
   ↓
ONNX / HEF export
   ↓
HEF testi
```

Daha fazla epoch her zaman daha iyi değildir. Test başarısı düşerken eğitimi uzatmak overfitting'i artırır; böyle bir durumda daha fazla ve daha çeşitli gerçek IR/saha verisi eklemek gerekir.